# Monitor: serial notebook orchestrator

Import this notebook as **nb_orchestrator** alongside the 19 transformation
notebooks. Attach the same default Lakehouse (for example **lh_monitor**) to
the orchestrator and every child. Match the names in NOTEBOOKS to the imported
Fabric notebook names; remove entries for domains you do not want to process.
Notebooks 05 (visualization) and 06 (controlled rerun validation) are excluded.
Notebook 22 is excluded unless INCLUDE_PIPELINE_ACTIVITY=True; import it first.
INCLUDE_SHORTCUTS=True includes retained shortcut evidence in 01-04, without
adding writers or tables. Use the same flag in rerun validation notebook 06.
INCLUDE_EVENTSTREAM_TOPOLOGY=True includes retained Eventstream configuration
in 01-04, also without new tables/writers. Match this flag in notebook 06.

Run manually with WRITE_TABLES=False first. After reviewing coverage, set it
to True to publish. The executing identity needs raw-data read access and,
for publication, curated-table write access. Shell credentials are not used.
The child notebooks must retain their marked parameter cells.

Execution is serial because concurrent same-prefix publication is unsupported
and domain notebooks share estate_coverage. Each child is attempted even after
another fails. The final summary identifies failures by notebook name; any
failure raises after the summary so a calling pipeline is marked failed.
Use Fabric's child-run snapshots for detailed diagnostics.

Succeeded means execution succeeded, not that collection coverage is complete
or fresh. Missing/failed collection may be reported without an exception.
The orchestrator makes no collection API calls and does not save a separate
status table. Review run outputs as potentially sensitive operational metadata.

After manual acceptance, call this notebook from a scheduled pipeline and verify
the pipeline execution identity. Run after collection has finished; avoid
overlapping orchestrator runs or other writers to the same table prefix.
Generated notebook: edit build_monitor_notebooks.py and regenerate, not these cells.


In [ ]:
# Attach the SAME default Lakehouse as the child notebooks.
RAW_ROOT = "/lakehouse/default/Files/monitor"
TABLE_PREFIX = "monitor_"
WRITE_TABLES = False  # Preview first; set True to publish Delta tables.
PERMISSION_MODE = "Scoped"  # Choose "Scoped" or "Tenant".
INCLUDE_PIPELINE_ACTIVITY = False  # Opt in after importing notebook 22.
INCLUDE_SHORTCUTS = False  # Opt in to shortcut evidence in 01-04.
INCLUDE_EVENTSTREAM_TOPOLOGY = False  # Opt in to topology evidence in 01-04.
NOTEBOOK_TIMEOUT_SECONDS = 3600
# Edit this list to match the imported notebook names in your workspace.
NOTEBOOKS = [
    "01_monitor_inventory_health",
    "02_monitor_permissions",
    "03_monitor_job_history",
    "04_monitor_lineage",
    "07_monitor_activity_events",
    "08_monitor_estate_catalog_schema",
    "09_monitor_connections",
    "10_monitor_gateways",
    "11_monitor_refresh_history",
    "12_monitor_apps",
    "13_monitor_capacity_performance",
    "14_monitor_directory_licensing",
    "15_monitor_workspace_roles",
    "16_monitor_tenant_settings",
    "17_monitor_domains",
    "18_monitor_git_connections",
    "19_monitor_tags",
    "20_monitor_broad_sharing",
    "21_monitor_deployment_pipelines",
]
if INCLUDE_PIPELINE_ACTIVITY:
    NOTEBOOKS.append("22_monitor_pipeline_activity")

In [ ]:
if (
    not isinstance(NOTEBOOKS, list)
    or not NOTEBOOKS
    or any(not isinstance(name, str) or not name.strip() for name in NOTEBOOKS)
    or len(set(NOTEBOOKS)) != len(NOTEBOOKS)
):
    raise ValueError("NOTEBOOKS must be a nonempty list of unique notebook names")
if type(WRITE_TABLES) is not bool:
    raise ValueError("WRITE_TABLES must be True or False")
if type(INCLUDE_SHORTCUTS) is not bool:
    raise ValueError("INCLUDE_SHORTCUTS must be True or False")
if type(INCLUDE_EVENTSTREAM_TOPOLOGY) is not bool:
    raise ValueError("INCLUDE_EVENTSTREAM_TOPOLOGY must be True or False")
if PERMISSION_MODE not in {"Scoped", "Tenant"}:
    raise ValueError("Choose Scoped or Tenant explicitly")
if type(NOTEBOOK_TIMEOUT_SECONDS) is not int or NOTEBOOK_TIMEOUT_SECONDS <= 0:
    raise ValueError("NOTEBOOK_TIMEOUT_SECONDS must be a positive integer")

print("Publication enabled." if WRITE_TABLES else "PREVIEW ONLY: no Delta writes.")
results = []
for name in NOTEBOOKS:
    print(f"Running {name}")
    try:
        notebookutils.notebook.run(
            name,
            NOTEBOOK_TIMEOUT_SECONDS,
            {
                "RAW_ROOT": RAW_ROOT,
                "TABLE_PREFIX": TABLE_PREFIX,
                "WRITE_TABLES": WRITE_TABLES,
                "PERMISSION_MODE": PERMISSION_MODE,
                "INCLUDE_SHORTCUTS": INCLUDE_SHORTCUTS,
                "INCLUDE_EVENTSTREAM_TOPOLOGY": INCLUDE_EVENTSTREAM_TOPOLOGY,
            },
        )
        results.append({"notebook": name, "status": "Succeeded", "error": ""})
    except Exception as exc:  # noqa: BLE001 - Attempt every child, then fail overall.
        results.append(
            {
                "notebook": name,
                "status": "Failed",
                "error": str(exc),
            }
        )

display(spark.createDataFrame(results, "notebook string, status string, error string"))
failed = [result["notebook"] for result in results if result["status"] == "Failed"]
if failed:
    raise RuntimeError("Failed notebooks: " + ", ".join(failed))
print("All selected notebooks succeeded. Review collection coverage separately.")